# Parameter Scan — Launch and Monitor

Defines and launches a parameter scan matrix using `ScanMatrix` from
`plasma_column.run_matrix`. Each (gas, method, pressure) combination
becomes an independent case directory under `runs/`.

**Scan axes covered**:
- Gas species: H2, Kr
- Gas pressure: 1e-6 → 3e-4 Torr (6 points, log-spaced)
- Method: seeded, callback *(add or remove in §1)*

> Set `DRY_RUN = True` to validate commands without launching WarpX.
> Set `DRY_RUN = False` to execute all cases sequentially.
> For parallel execution, use `scripts/run_scan.py` with a job scheduler.


In [ ]:
from pathlib import Path
import os, sys, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate project root
_ROOT = Path.cwd()
while _ROOT.name != 'plasma_column' and _ROOT.parent != _ROOT:
    _ROOT = _ROOT.parent
if str(_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(_ROOT / 'src'))

WORK           = Path.home() / 'Work' / 'simulation_codes-working'
WARPX_DATA_DIR = WORK / 'warpx-data'
RUNS_DIR       = _ROOT / 'runs'
PLOTS_DIR      = _ROOT / 'plots'
RUNS_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

os.environ['WARPX_DATA_DIR']  = str(WARPX_DATA_DIR)
os.environ['LD_LIBRARY_PATH'] = (
    str(WORK / 'warpx' / 'install' / 'lib') + ':'
    + os.environ.get('LD_LIBRARY_PATH', '')
)
print('Python :', sys.executable)
print('ROOT   :', _ROOT)
print('WarpX data:', WARPX_DATA_DIR)


In [ ]:
from plasma_column.notebook_utils import print_simulation_config
from plasma_column.run_matrix import ScanMatrix, ScanParameter, build_scan_dataframe, run_scan_matrix

_DEFAULTS = {
    'scan_name':       'pressure_scan_h2_kr',
    'gases':           'H2, Kr',
    'pressures [Torr]':'1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4',
    'methods':         'seeded, callback',
    'beam energy [keV]':30.0,
    'beam current [mA]':10.0,
    'max_steps (smoke)':500,
    'max_steps (full)': 120000,
    'grid (seeded)':   '32 x 32 x 256',
    'grid (callback)': '24 x 24 x 128',
}
_OVERRIDES = {}  # e.g. restrict to one gas or fewer pressures
print_simulation_config(
    notebook_title='Parameter Scan — Launch and Monitor',
    defaults=_DEFAULTS, overrides=_OVERRIDES,
)


## 1. Define scan matrix


In [ ]:
DRY_RUN = True   # change to False to run simulations

SCRIPT_SEEDED   = _ROOT / 'plasma_column_mcc_picmi_v7.py'
SCRIPT_CALLBACK = _ROOT / 'plasma_column_callback_source_picmi_v3.py'

# Pressure axis (6 log-spaced points)
PRESSURES = [1e-6, 3e-6, 1e-5, 3e-5, 1e-4, 3e-4]  # Torr

# Smoke-test settings (small grid, few steps)
SMOKE_MAX_STEPS = 500
PROD_MAX_STEPS  = 120000

# Switch between smoke test and production:
MAX_STEPS = SMOKE_MAX_STEPS  # → change to PROD_MAX_STEPS for real runs

scan_seeded = ScanMatrix(
    scan_name  = 'pressure_scan',
    script     = SCRIPT_SEEDED,
    parameters = [ScanParameter('pressure_torr', PRESSURES)],
    fixed      = {
        'neutralization': '0.5',
        'mcc':            'electron_impact',
        'plasma_age':     '2e-4',
        'max_steps':       MAX_STEPS,
        'diag_period':     max(100, MAX_STEPS // 24),
        'reduced_diag_period': 100,
        'nx': 32, 'ny': 32, 'nz': 256,
    },
    gases   = ['H2', 'Kr'],
    methods = ['seeded'],
    dry_run = DRY_RUN,
    runs_root = RUNS_DIR,
)

scan_callback = ScanMatrix(
    scan_name  = 'callback_scan',
    script     = SCRIPT_CALLBACK,
    parameters = [ScanParameter('pressure_torr', PRESSURES)],
    fixed      = {
        'max_steps':              min(20000, MAX_STEPS),
        'diag_period':            max(100, min(20000, MAX_STEPS) // 4),
        'reduced_diag_period':    100,
        'source_every_n_steps':   10,
        'enable_ionization_source': 1,
        'nx': 24, 'ny': 24, 'nz': 128,
    },
    gases   = ['H2', 'Kr'],
    methods = ['callback'],
    dry_run = DRY_RUN,
    runs_root = RUNS_DIR,
)

df_seeded   = build_scan_dataframe(scan_seeded)
df_callback = build_scan_dataframe(scan_callback)
df_all_cases = pd.concat([df_seeded, df_callback], ignore_index=True)
print(f'Total cases: {len(df_all_cases)}')
df_all_cases[['case_name','gas','method','pressure_torr']]


## 2. Validate scan with YAML launcher (dry-run)


In [ ]:
import subprocess
# Validate using the YAML-driven script/run_scan.py (no simulation execution)
result = subprocess.run(
    [sys.executable, 'scripts/run_scan.py',
     '--matrix', 'cases/pressure_scan_h2_kr.yaml', '--dry_run'],
    capture_output=True, text=True, cwd=str(_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


## 3. Launch scan

Uncomment the `run_scan_matrix()` calls to execute cases sequentially.
Each call blocks until all cases for that matrix finish.


In [ ]:
# ── Seeded pressure scan ─────────────────────────────────────────────────
# results_seeded = run_scan_matrix(
#     df_seeded, scan_seeded,
#     warpx_data_dir=WARPX_DATA_DIR,
# )

# ── Callback pressure scan ───────────────────────────────────────────────
# results_callback = run_scan_matrix(
#     df_callback, scan_callback,
#     warpx_data_dir=WARPX_DATA_DIR,
# )

print('Launch cells ready — uncomment run_scan_matrix() calls to start.')


## 4. Monitor progress


In [ ]:
# Count how many cases have completed diagnostic files
from plasma_column.run_matrix import _find_diag

completed, pending = [], []
for _, row in df_all_cases.iterrows():
    d = RUNS_DIR / row['case_name']
    if _find_diag(d) is not None:
        completed.append(row['case_name'])
    else:
        pending.append(row['case_name'])

print(f'Completed: {len(completed)}/{len(df_all_cases)}')
for c in completed: print(f'  ✓ {c}')
print(f'Pending:   {len(pending)}')
for p in pending:   print(f'  … {p}')
